In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import matplotlib.pyplot as plt

np.random.seed(42)


SPLIT_DIR = Path("../backend/data/splits_70_15_15")
KF_DIR    = Path("../backend/data/kf_outputs")

VAL_PATH  = SPLIT_DIR / "val.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

KF_TS_PARQUET = KF_DIR / "kf_fair_val_test_forecast_nis.parquet"
KF_SUMMARY_CSV = KF_DIR / "kf_fair_summary_metrics_forecast.csv"

OUT_VAL_CSV = KF_DIR / "kf_hive_drop_audit_val.csv"
OUT_TEST_CSV = KF_DIR / "kf_hive_drop_audit_test.csv"
OUT_COMBINED_CSV = KF_DIR / "kf_hive_drop_audit_combined.csv"
OUT_SUMMARY_JSON = KF_DIR / "kf_hive_drop_audit_summary.json"

PLOTS_DIR = KF_DIR / "audit_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------
# Settings (match KF notebook)
# --------------------
OBS_COLS = ["temperature", "humidity", "audio_density"]
MIN_USABLE_ROWS = 200   # MUST match your KF evaluation threshold

# --------------------
# Load splits
# --------------------
def clean_split(df):
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"]).sort_values(["tag_number","published_at"])
    return df.reset_index(drop=True)

print("Loading splits...")
val_df  = clean_split(pd.read_parquet(VAL_PATH))
test_df = clean_split(pd.read_parquet(TEST_PATH))

print("Val:", val_df.shape, "Test:", test_df.shape)
print("Hives in VAL:", val_df["tag_number"].nunique(), "Hives in TEST:", test_df["tag_number"].nunique())

# --------------------
# Load KF outputs
# --------------------
print("\nLoading KF per-timestep parquet:", KF_TS_PARQUET)
kf_ts = pd.read_parquet(KF_TS_PARQUET)
kf_ts["published_at"] = pd.to_datetime(kf_ts["published_at"], utc=True, errors="coerce")
kf_ts["hive_id"] = pd.to_numeric(kf_ts["hive_id"], errors="coerce").astype("Int64")
kf_ts = kf_ts.dropna(subset=["published_at","hive_id"]).sort_values(["hive_id","published_at"])

print("KF timestep rows:", kf_ts.shape, "| splits:", kf_ts["split"].unique())

print("\nLoading KF summary csv:", KF_SUMMARY_CSV)
kf_sum = pd.read_csv(KF_SUMMARY_CSV)
kf_sum["hive_id"] = pd.to_numeric(kf_sum["hive_id"], errors="coerce").astype("Int64")
kf_sum = kf_sum.dropna(subset=["hive_id"])
print("KF summary rows:", kf_sum.shape)

# --------------------
# Helper metrics
# --------------------
def usable_rows_any_sensor(df_h):
    z = df_h[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def obs_rates(df_h):
    out = {}
    out["obs_rate_any"] = float(np.isfinite(df_h[OBS_COLS].to_numpy(float)).any(axis=1).mean())
    for c in OBS_COLS:
        out[f"obs_rate_{c}"] = float(np.isfinite(df_h[c].to_numpy(float)).mean())
    return out

def audit_split(split_name, split_df):
    """
    For each hive in split_df:
    - Check usable rows (any observed sensor)
    - Check if predictions exist in KF parquet for that split
    - Check timestamp alignment between split and KF outputs
    - Check if hive appears in KF summary CSV
    - Produce explicit drop reasons
    """
    assert split_name in ("val","test")
    split_df = split_df.copy()
    split_df["hive_id"] = split_df["tag_number"].astype("Int64")

    # KF per-timestep subset
    kf_s = kf_ts[kf_ts["split"] == split_name].copy()

    all_hives = sorted(split_df["hive_id"].dropna().unique().tolist())

    rows = []
    for hive_id in all_hives:
        g = split_df[split_df["hive_id"] == hive_id]
        n_rows_split = int(len(g))
        usable_any = usable_rows_any_sensor(g)
        rates = obs_rates(g)

        # predictions for this hive?
        p = kf_s[kf_s["hive_id"] == hive_id]
        n_rows_pred = int(len(p))

        # timestamp match rate
        # (how many split timestamps exist in predictions)
        if n_rows_pred > 0:
            ts_true = pd.Index(g["published_at"])
            ts_pred = pd.Index(p["published_at"])
            n_match = int(ts_true.isin(ts_pred).sum())
            match_rate = float(n_match / max(len(ts_true), 1))
        else:
            n_match = 0
            match_rate = np.nan

        # merge count (ground truth × prediction intersection)
        if n_rows_pred > 0:
            merged = g[["hive_id","published_at"]].merge(
                p[["hive_id","published_at"]],
                on=["hive_id","published_at"],
                how="inner"
            )
            n_rows_merged = int(len(merged))
        else:
            n_rows_merged = 0

        in_summary = bool((kf_sum["hive_id"] == hive_id).any())

        # drop reasons (explicit string list)
        reasons = []
        if usable_any < MIN_USABLE_ROWS:
            reasons.append(f"usable_rows<{MIN_USABLE_ROWS}")
        if n_rows_pred == 0:
            reasons.append("no_predictions_in_kf_parquet")
        if (n_rows_pred > 0) and (match_rate < 0.98):
            reasons.append("timestamp_mismatch(<0.98)")
        if not in_summary:
            reasons.append("missing_from_kf_summary_csv")

        drop_reason = "OK" if len(reasons) == 0 else "|".join(reasons)

        rows.append({
            "split": split_name,
            "hive_id": int(hive_id),
            "drop_reason": drop_reason,
            "in_kf_summary_csv": in_summary,
            "n_rows_split": n_rows_split,
            "usable_rows_any": usable_any,
            "n_rows_pred": n_rows_pred,
            "n_rows_merged": n_rows_merged,
            "timestamp_match_rate": match_rate,
            **rates,
            "n_obs_any": int(rates["obs_rate_any"] * n_rows_split),
            "n_obs_temperature": int(rates["obs_rate_temperature"] * n_rows_split),
            "n_obs_humidity": int(rates["obs_rate_humidity"] * n_rows_split),
            "n_obs_audio_density": int(rates["obs_rate_audio_density"] * n_rows_split),
            "start": str(g["published_at"].min()),
            "end": str(g["published_at"].max()),
        })

    audit = pd.DataFrame(rows).sort_values(["split","drop_reason","hive_id"]).reset_index(drop=True)
    return audit

# --------------------
# Run audits
# --------------------
audit_val  = audit_split("val", val_df)
audit_test = audit_split("test", test_df)

print("\nVAL audit rows:", len(audit_val), "| OK:", int((audit_val['drop_reason']=="OK").sum()))
print("TEST audit rows:", len(audit_test), "| OK:", int((audit_test['drop_reason']=="OK").sum()))

# Save CSVs
audit_val.to_csv(OUT_VAL_CSV, index=False)
audit_test.to_csv(OUT_TEST_CSV, index=False)
combined = pd.concat([audit_val, audit_test], ignore_index=True)
combined.to_csv(OUT_COMBINED_CSV, index=False)

print("\nSaved:")
print(" -", OUT_VAL_CSV)
print(" -", OUT_TEST_CSV)
print(" -", OUT_COMBINED_CSV)

def summarize(audit_df):
    total = int(len(audit_df))
    ok = int((audit_df["drop_reason"] == "OK").sum())
    not_ok = total - ok
    reasons = audit_df["drop_reason"].value_counts().to_dict()
    return {"total_hives": total, "ok": ok, "dropped_or_flagged": not_ok, "reason_counts": reasons}

summary = {
    "min_usable_rows_rule": int(MIN_USABLE_ROWS),
    "val": summarize(audit_val),
    "test": summarize(audit_test),
}
with open(OUT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(" -", OUT_SUMMARY_JSON)


def plot_reason_counts(audit_df, split_name):
    vc = audit_df["drop_reason"].value_counts()
    plt.figure()
    vc.plot(kind="bar")
    plt.title(f"{split_name.upper()}: Hive drop reasons (explicit)")
    plt.ylabel("# hives")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    out = PLOTS_DIR / f"{split_name}_drop_reason_counts.png"
    plt.savefig(out, dpi=200)
    plt.show()
    print("Saved plot:", out)

def plot_match_rate(audit_df, split_name):
    x = audit_df["timestamp_match_rate"].dropna().to_numpy()
    plt.figure()
    plt.hist(x, bins=20)
    plt.title(f"{split_name.upper()}: timestamp match rate distribution")
    plt.xlabel("match rate (split timestamps found in KF predictions)")
    plt.ylabel("# hives")
    plt.tight_layout()
    out = PLOTS_DIR / f"{split_name}_timestamp_match_rate_hist.png"
    plt.savefig(out, dpi=200)
    plt.show()
    print("Saved plot:", out)

def plot_obs_rate_any(audit_df, split_name):
    x = audit_df["obs_rate_any"].to_numpy()
    plt.figure()
    plt.hist(x, bins=20)
    plt.title(f"{split_name.upper()}: observation availability (any sensor)")
    plt.xlabel("obs_rate_any")
    plt.ylabel("# hives")
    plt.tight_layout()
    out = PLOTS_DIR / f"{split_name}_obs_rate_any_hist.png"
    plt.savefig(out, dpi=200)
    plt.show()
    print("Saved plot:", out)

plot_reason_counts(audit_val, "val")
plot_reason_counts(audit_test, "test")
plot_match_rate(audit_val, "val")
plot_match_rate(audit_test, "test")
plot_obs_rate_any(audit_val, "val")
plot_obs_rate_any(audit_test, "test")


missing_val = audit_val[audit_val["in_kf_summary_csv"] == False]["hive_id"].tolist()
missing_test = audit_test[audit_test["in_kf_summary_csv"] == False]["hive_id"].tolist()

print("\nVAL: hives NOT in kf_summary_metrics_forecast.csv ->", len(missing_val))
print(missing_val[:50])

print("\nTEST: hives NOT in kf_summary_metrics_forecast.csv ->", len(missing_test))
print(missing_test[:50])


Loading splits...
Val: (120160, 31) Test: (120187, 31)
Hives in VAL: 52 Hives in TEST: 52

Loading KF per-timestep parquet: ..\backend\data\kf_outputs\kf_fair_val_test_forecast_nis.parquet
KF timestep rows: (118009, 20) | splits: <ArrowStringArray>
['val', 'test']
Length: 2, dtype: str

Loading KF summary csv: ..\backend\data\kf_outputs\kf_fair_summary_metrics_forecast.csv


FileNotFoundError: [Errno 2] No such file or directory: '..\\backend\\data\\kf_outputs\\kf_fair_summary_metrics_forecast.csv'